# MLPRouter - Training

This notebook demonstrates how to train the **MLPRouter** (Multi-Layer Perceptron Router).

## Overview

MLPRouter uses a neural network classifier with multiple hidden layers to route queries.

**Key Features**:
- Can learn complex non-linear decision boundaries
- Flexible architecture with configurable layers
- Good for large-scale routing problems

## 1. Environment Setup

In [ ]:
# Install required packages (for Colab)
!pip install llmrouter-lib transformers torch
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
from llmrouter.models.mlprouter import MLPRouter, MLPTrainer
from llmrouter.utils import setup_environment

setup_environment()
print("Environment setup complete!")

## 2. Configuration

MLPRouter uses the following configuration parameters:

| Parameter | Description | Default |
|-----------|-------------|--------|
| `hidden_layer_sizes` | Neurons in each hidden layer | [128, 64] |
| `activation` | Activation function | "relu" |
| `solver` | Optimizer: "adam", "lbfgs", "sgd" | "adam" |
| `alpha` | L2 regularization | 0.0001 |
| `learning_rate` | Learning rate schedule | "adaptive" |
| `max_iter` | Maximum iterations | 500 |

In [ ]:
import yaml

CONFIG_PATH = "configs/model_config_train/mlprouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

## 3. Initialize Router

In [ ]:
router = MLPRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Number of training samples: {len(router.routing_data_train)}")
print(f"Number of LLM candidates: {len(router.llm_data)}")
print(f"LLM candidates: {list(router.llm_data.keys())}")

In [ ]:
# Inspect MLP architecture
print("MLP Model Parameters:")
print(router.mlp_model.get_params())

## 4. Training

In [ ]:
trainer = MLPTrainer(router=router, device='cpu')

print("Trainer initialized!")
print(f"Training samples: {len(trainer.query_embeddings)}")
print(f"Save path: {trainer.save_model_path}")

In [ ]:
print("Starting training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training completed!")

## 5. Model Verification

In [ ]:
from llmrouter.utils import load_model
import numpy as np

saved_model = load_model(trainer.save_model_path)

print("Model loaded successfully!")
print(f"Model type: {type(saved_model).__name__}")
print(f"Number of layers: {len(saved_model.hidden_layer_sizes)}")
print(f"Layer sizes: {saved_model.hidden_layer_sizes}")
print(f"Classes: {saved_model.classes_}")

In [ ]:
# Quick prediction test
test_embedding = trainer.query_embeddings[0].reshape(1, -1)
prediction = saved_model.predict(test_embedding)

print(f"Test prediction: {prediction[0]}")

proba = saved_model.predict_proba(test_embedding)
print(f"\nPrediction probabilities:")
for model, prob in zip(saved_model.classes_, proba[0]):
    print(f"  {model}: {prob:.4f}")

## 6. Learning Curve Analysis

In [ ]:
import matplotlib.pyplot as plt

# Plot training loss curve
if hasattr(saved_model, 'loss_curve_'):
    plt.figure(figsize=(10, 5))
    plt.plot(saved_model.loss_curve_)
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.title('MLP Training Loss Curve')
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("Loss curve not available (model may use 'lbfgs' solver)")

## 7. Architecture Comparison

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

X = np.array(trainer.query_embeddings)
y = np.array(trainer.model_name_list)

# Test different architectures
architectures = [
    (64,),
    (128,),
    (128, 64),
    (256, 128),
    (256, 128, 64),
]

print("Architecture comparison:")
print("=" * 50)

results = []
for arch in architectures:
    mlp = MLPClassifier(hidden_layer_sizes=arch, max_iter=200, random_state=42)
    scores = cross_val_score(mlp, X, y, cv=3, scoring='accuracy')
    results.append((arch, scores.mean(), scores.std()))
    print(f"{str(arch):20} Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

best_arch, best_score, _ = max(results, key=lambda x: x[1])
print(f"\nBest architecture: {best_arch} with accuracy: {best_score:.4f}")

## Summary

In this notebook, we:

1. **Loaded Configuration**: Set up MLPRouter with YAML configuration
2. **Trained Model**: Used MLPRouterTrainer to fit the neural network
3. **Verified Model**: Loaded and tested the saved model
4. **Compared Architectures**: Found optimal layer configuration

**Next Steps**:
- Use the next part of notebook for inference
- Experiment with different activation functions

# MLPRouter - Inference

This part of notebook demonstrates how to use a trained **MLPRouter** for inference.

## 1. Environment Setup (optional)

In [ ]:
# Install required packages (for Colab)
!pip install llmrouter-lib transformers torch
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [ ]:
from llmrouter.models.mlprouter import MLPRouter
from llmrouter.utils import setup_environment, load_model, get_longformer_embedding
import yaml

setup_environment()

## 2. Load Trained Router

In [ ]:
CONFIG_PATH = "configs/model_config_train/mlprouter.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

config['model_path']['load_model_path'] = config['model_path']['save_model_path']

INFERENCE_CONFIG_PATH = "configs/model_config_test/mlprouter_inference.yaml"
os.makedirs(os.path.dirname(INFERENCE_CONFIG_PATH), exist_ok=True)

with open(INFERENCE_CONFIG_PATH, 'w') as f:
    yaml.dump(config, f)

router = MLPRouter(yaml_path=INFERENCE_CONFIG_PATH)
print(f"Router loaded with {len(router.llm_data)} LLM candidates")

## 3. Single Query Routing

In [ ]:
EXAMPLE_QUERIES = [
    {"query": "What is the capital of France?"},
    {"query": "Solve the equation: 2x + 5 = 15"},
    {"query": "Write a Python function to check if a number is prime."},
    {"query": "Explain quantum computing in simple terms."},
]

print("Routing Results:")
print("=" * 60)

for i, query in enumerate(EXAMPLE_QUERIES, 1):
    result = router.route_single(query)
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result['model_name']}")

## 4. Confidence Analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

model_path = os.path.join(PROJECT_ROOT, config['model_path']['load_model_path'])
mlp_model = load_model(model_path)

def get_confidence(query_text):
    embedding = get_longformer_embedding(query_text).numpy().reshape(1, -1)
    proba = mlp_model.predict_proba(embedding)[0]
    prediction = mlp_model.predict(embedding)[0]
    confidence = max(proba)
    return prediction, confidence, dict(zip(mlp_model.classes_, proba))

print("Confidence Analysis:")
print("=" * 60)
for query in EXAMPLE_QUERIES:
    pred, conf, probs = get_confidence(query['query'])
    print(f"Query: {query['query'][:40]}...")
    print(f"  Prediction: {pred} (Confidence: {conf:.4f})")

## 5. File-Based Inference

Load queries from a file and save results.

In [ ]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/mlprouter_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")
    
    # Save results
    save_results_to_file(file_results, OUTPUT_FILE)
    
    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

## Summary

This notebook demonstrated:
1. Loading a trained MLPRouter
2. Single query routing
3. Confidence analysis

MLPRouter is effective for:
- Complex decision boundaries
- Large-scale routing with many LLMs